# 🧪 Sesión 12 · Inyección de Dependencias (FastAPI Depends / Contenedor IoC)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo cambias una dependencia sin tocar el endpoint?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/diagnostico
HTTP/1.1 200 OK
{"repositorio":"RepositorioMemoria"}

POST /api/libros
{"titulo":"Rayuela","anio":1963}
HTTP/1.1 201 Created
{"id":1,"titulo":"Rayuela","anio":1963}

GET /api/libros
HTTP/1.1 200 OK
{"libros":[{"id":1,"titulo":"Rayuela","anio":1963}]}
```

_El endpoint no crea el repositorio: lo recibe. Esa es la diferencia entre acoplarse y depender de una interfaz._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Inyectar un repositorio con Depends de FastAPI.
- Sustituir una dependencia sin reescribir el endpoint.
- Verificar el efecto de la inyección en el contrato HTTP.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El diagnóstico revela el repositorio inyectado → HTTP 200
- [ ] El endpoint inyectado crea un libro → HTTP 201
- [ ] El repositorio conserva el estado → HTTP 200
- [ ] La validación sigue en el borde → HTTP 422
- [ ] El endpoint usa Depends
- [ ] Existe la función de inyección

## 🧩 El reto

Implementa un repositorio en memoria y inyéctalo en los endpoints con Depends. El diagnóstico debe evidenciar qué implementación se está usando.

**Requisitos del contrato**

- Existe una función `get_repositorio()` que devuelve una instancia de `RepositorioMemoria`.
- Los endpoints de libros reciben el repositorio con `Depends(get_repositorio)`.
- GET /api/diagnostico responde {"repositorio": "RepositorioMemoria"}.
- El CRUD mantiene el contrato: GET 200, POST 201, datos inválidos 422.

> **Pista conceptual:** Si el endpoint hace `RepositorioMemoria()` dentro de la función, no está inyectando: está acoplando. La dependencia debe llegar como parámetro.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion12.md` explicando qué dependencia inyectaste.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tu endpoint recibirá el repositorio inyectado")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'El diagnóstico revela el repositorio inyectado',
  'metodo': 'GET',
  'ruta': '/api/diagnostico',
  'estado': 200,
  'json_exacto': {'repositorio': 'RepositorioMemoria'}},
 {'descripcion': 'El endpoint inyectado crea un libro',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'El repositorio conserva el estado',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}]}},
 {'descripcion': 'La validación sigue en el borde',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 'malo'},
  'estado': 422}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", exigir=["Depends"], descripcion="El endpoint usa Depends"))
resultados.append(revisar_archivo("app.py", exigir=["get_repositorio"], descripcion="Existe la función de inyección"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion12.md` a `auditorias/auditoria_sesion12.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
